# 00 — Keşifsel veri analizi
Sınıf dağılımı, eksik veri, sınıf bazında log değerleri, örnek kuyu grafiği.

In [ ]:
# --- Kurulum: Colab'da repoyu klonlar/günceller, yerelde repo köküne geçer ---
import os, sys, pathlib
REPO_URL = "https://github.com/aysenursimsk/force2020-litoloji.git"   # ← kendi repo adresiniz

if "google.colab" in sys.modules:
    if not pathlib.Path("/content/force2020-litoloji").exists():
        !git clone -q {REPO_URL} /content/force2020-litoloji
    %cd /content/force2020-litoloji
    !git pull -q

root = pathlib.Path.cwd()
if not (root / "src").exists():
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from src import config as C, veri, degerlendir as D

train, test = veri.yukle()
d = veri.temizle(train)
print(train.shape, "| eğitim kuyusu:", train.WELL.nunique(), "|| test:", test.shape, "| test kuyusu:", test.WELL.nunique())

## Sınıf dağılımı (dengesizlik)

In [ ]:
names = d.y.map(dict(enumerate(C.CLASS_NAMES)))
tab = pd.DataFrame({"satır": names.value_counts(),
                    "yüzde": (names.value_counts(normalize=True) * 100).round(2),
                    "kuyu sayısı": d.groupby(names).WELL.nunique()})
display(tab)
tab["satır"].sort_values().plot.barh(logx=True, figsize=(7, 4), title="Sınıf başına satır (log ölçek)")
plt.show()

## Eksik veri

In [ ]:
miss = (train.isna().mean() * 100).sort_values(ascending=False)
miss[miss > 0].plot.bar(figsize=(10, 3), title="Sütun başına eksik veri (%)"); plt.show()

## Sınıf bazında log medyanları
Yorum için: şeylde GR yüksek, halitte RHOB düşük, anhidritte RHOB yüksek…

In [ ]:
d.groupby(names)[["GR", "RHOB", "NPHI", "DTC", "PEF", "RDEP"]].median().round(2)

## Örnek kuyu grafiği

In [ ]:
D.kuyu_grafigi(d, d.WELL.iloc[0])